<a href="https://colab.research.google.com/github/oliviadellaglio/ds2002-fa26/blob/main/2026_10_09_%E2%80%94_Public_API_to_Clean_Table_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [8]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    last_error = None
    for attempt in range(tries):
      try:
        #send GET request; give up after 10 seconds
        r = requests.get(BASE, params=params, timeout=10)
        #Network problem, no status code so save error and retry logic below
      except (requests.ConnectionError, requests.Timeout) as e:
        last_error = e
      else:
        #if request completed, inspect status code
        if r.status_code == 200:
          #success, JSON has "daily" dict of equal length lists which pandas can turn to table
          return pd.DataFrame(r.json()['daily'])

        if r.status_code not in RETRY_STATUSES:
        # Permanent error, retrying won't help, so stop immediately
          raise RuntimeError(
              f'Open-Meteo returned {r.status_code}: {r.text[:200]}'
          )

        #Temporary error: record and go around loop again
        last_error = RuntimeError(f'Open-Meteo returned {r.status_code}')

      if attempt < tries - 1:
        time.sleep(2**attempt)

    #Every attempt failed, chain last error
    raise RuntimeError(f"All {tries} attempts failed") from last_error

    pass

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [30]:
weekends = ['2026-09-05', '2026-10-03']
used_fallback = False

try:
  #
  df_range = get_weather(min(weekends), max(weekends))

  #keept only the two weekend rows and hold date as string
  home_weather = df_range[df_range['time'].isin(weekends)].reset_index(drop=True)
  assert len(home_weather) == 2, f'expected 2 rows, got {len(home_weather)}'

except RuntimeError as e:
  #get_weather only raises RuntimeError after permanent arror or exhausted retries
  used_fallback = True
  print(f'API call failed, switching to Part 3 fallback: {e}')
  home_weather = None

print('used_fallback =', used_fallback)
print(home_weather)

used_fallback = False
         time  temperature_2m_max  precipitation_sum
0  2026-09-05                29.4                2.5
1  2026-10-03                22.0               14.7


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [13]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if home_weather is None:
#API call failed, record and use sample data
  used_fallback = True
  wx = FALLBACK.copy() #copy so FALLBACK never modified
  wx['source'] = 'fallback sample'

else:
  #API call succeeds
  wx = home_weather.copy()
  wx['source'] = 'open-meteo'

print('used_fallback=', used_fallback)
print(wx)

used_fallback= False
         time  temperature_2m_max  precipitation_sum      source
0  2026-09-05                29.4                2.5  open-meteo
1  2026-10-03                22.0               14.7  open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [18]:
#rename columns to my names
wx = wx.rename(columns={
  'time': 'date',
  'temperature_2m_max': 'temp_max',
  'precipitation_sum': 'precip_mm',
})

#convery raw date to string to join on dates
wx['date'] = pd.to_datetime(wx['date'])

#keep only 4 columns
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

#show column types
print(wx.dtypes)
print(wx)

date         datetime64[ns]
temp_max            float64
precip_mm           float64
source               object
dtype: object
        date  temp_max  precip_mm      source
0 2026-09-05      29.4        2.5  open-meteo
1 2026-10-03      22.0       14.7  open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [28]:
#assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
#assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
#assert pd.api.types.is_datetime64_any_dtype(wx['date'])
#assert wx['precip_mm'].notna().all(), 'missing precipitation'
#assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [32]:
#get wettest day
wet = wx.loc[wx['precip_mm'].idxmax()]
#get dryest day
dry = wx.loc[wx['precip_mm'].idxmin()]

#find difference and print
diff_mm = wet['precip_mm'] - dry['precip_mm']
print(f"{wet['date'].date()} was wettest ({wet['precip_mm']:.1f} mm), "
      f"{dry['date'].date()} was driest ({dry['precip_mm']:.1f} mm). "
      f"Difference: {diff_mm:.1f} mm.")


2026-10-03 was wettest (14.7 mm), 2026-09-05 was driest (2.5 mm). Difference: 12.2 mm.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [38]:
from datetime import datetime
from zoneinfo import ZoneInfo


LAT, LON = 38.03, -78.51   # same defaults get_weather used in Part 2
pulled_at = datetime.now(ZoneInfo('America/New_York'))

print('Endpoint     :', BASE)
print('Coordinates  :', f'{LAT}, {LON}')
print('Date range   :', f"{min(weekends)} to {max(weekends)}")
print('Pulled at    :', pulled_at.strftime('%Y-%m-%d %H:%M %Z'))
print('Data source  :', 'FALLBACK SAMPLE (not real observations)' if used_fallback
      else 'REAL (Open-Meteo archive API)')

Endpoint     : https://archive-api.open-meteo.com/v1/archive
Coordinates  : 38.03, -78.51
Date range   : 2026-09-05 to 2026-10-03
Pulled at    : 2026-10-11 23:17 EDT
Data source  : REAL (Open-Meteo archive API)


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

The API failes, get_weather retries and runs out of attempts. pd.DataFrame() is returned and no error raused. used_fallback is never changed because the code never runs after it times out. If it is merged onto 200 days of sales, sales rows are kept but rain column is blank. The chart will draw without error, but there will be no rain points since the column is blank, creating a flat line. This is difficult to catch because the code runs without error. Pandas and charts skip the missing values and still produce a chart.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [35]:
orl_raw = get_weather(min(weekends), max(weekends), lat=28.54, lon=-81.38)

#keep two weekend rows and apply the same renaming as Part 4
orl = (orl_raw[orl_raw['time'].isin(weekends)]
       .rename(columns={'time': 'date',
                        'temperature_2m_max': 'temp_max',
                        'precipitation_sum': 'precip_mm'})
       .copy())
orl['date'] = pd.to_datetime(orl['date'])
orl['source'] = 'open-meteo'
orl['city'] = 'Orlando'

#label charlottesville
cville = wx.copy()
cville['city'] = 'Charlottesville'

# Stack into one long-format frame: one row per city per date
cols = ['city', 'date', 'temp_max', 'precip_mm', 'source']
weather_long = pd.concat([cville[cols], orl[cols]], ignore_index=True)

print(weather_long)

              city       date  temp_max  precip_mm      source
0  Charlottesville 2026-09-05      29.4        2.5  open-meteo
1  Charlottesville 2026-10-03      22.0       14.7  open-meteo
2          Orlando 2026-09-05      32.1        3.9  open-meteo
3          Orlando 2026-10-03      30.4       14.5  open-meteo
